Stencil system

Install the wheel using pip 

In [17]:
import subprocess, sys, glob, pathlib

# location of .whl file
_search_paths = [
    pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path("."),
    pathlib.Path("../../build/dist"),
]
_wheel = next(
    (str(w) for p in _search_paths for w in p.glob("stencil_lib-*.whl")),
    None
)
if _wheel is None:
    raise FileNotFoundError("stencil_lib wheel not found. Run 'inv build' or place the wheel alongside the notebook.")

# pip install the wheel
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       _wheel, "--force-reinstall"])


print(f"stencil_lib installed from: {_wheel}")


stencil_lib installed from: ..\..\build\dist\stencil_lib-0.1.0-py3-none-any.whl


Start using the stencil_lib library

In [18]:
# Start using the library
from stencil_lib import CipherConfig, cipher_encrypt, cipher_decrypt

In [19]:
class AESConfig(CipherConfig):
    algo = "aes"
    def __init__(self, key: bytes, mode: int, **mode_params):
        self.parameters = {
            "key": key,
            "mode": mode,
            "mode_params": mode_params
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.encrypt(plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.decrypt(ciphertext)


In [20]:
class CaesarConfig(CipherConfig):
    algo = "caesar"
    def __init__(self, shift: int):
        self.parameters = {
            "shift": shift
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b + shift) % 256 for b in plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b - shift) % 256 for b in ciphertext)


In [21]:
class VigenereConfig(CipherConfig):
    algo = "vigenere"
    def __init__(self, keyword: bytes):
        self.parameters = {
            "keyword": keyword
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b + key[i % key_len]) % 256 for i, b in enumerate(plaintext))
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b - key[i % key_len]) % 256 for i, b in enumerate(ciphertext))


## Demo — Caesar Cipher


In [22]:
plaintext = b"HELLO WORLD"

cfg = CaesarConfig(shift=13)
ciphertext = cipher_encrypt(plaintext, cfg)
recovered  = cipher_decrypt(ciphertext, cfg)

print(f"Plaintext : {plaintext}")
print(f"Ciphertext: {ciphertext}")
print(f"Recovered : {recovered}")
assert recovered == plaintext
print("Caesar round-trip OK")


Plaintext : b'HELLO WORLD'
Ciphertext: b'URYY\\-d\\_YQ'
Recovered : b'HELLO WORLD'
Caesar round-trip OK


## Demo — Vigenère Cipher


In [23]:
plaintext = b"HELLO WORLD"

cfg = VigenereConfig(keyword=b"SECRET")
ciphertext = cipher_encrypt(plaintext, cfg)
recovered  = cipher_decrypt(ciphertext, cfg)

print(f"Plaintext : {plaintext}")
print(f"Ciphertext: {ciphertext}")
print(f"Recovered : {recovered}")
assert recovered == plaintext
print("Vigenère round-trip OK")


Plaintext : b'HELLO WORLD'
Ciphertext: b'\x9b\x8a\x8f\x9e\x94t\xaa\x94\x95\x9e\x89'
Recovered : b'HELLO WORLD'
Vigenère round-trip OK


## Demo — AES Cipher (CBC mode)


In [24]:
import os
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad

plaintext = b"HELLO WORLD"                 # 11 bytes — needs padding for CBC
raw_key   = os.urandom(16)                 # 128-bit key (AES-128)
iv        = os.urandom(16)                 # random IV

cfg = AESConfig(key=raw_key, mode=AES.MODE_CBC, iv=iv)

ciphertext = cipher_encrypt(pad(plaintext, AES.block_size), cfg)
recovered  = unpad(cipher_decrypt(ciphertext, cfg), AES.block_size)

print(f"Plaintext : {plaintext}")
print(f"Ciphertext: {ciphertext.hex()}")
print(f"Recovered : {recovered}")
assert recovered == plaintext
print("AES-CBC round-trip OK")


Plaintext : b'HELLO WORLD'
Ciphertext: 9cda62a23855e140968e1384fa816109
Recovered : b'HELLO WORLD'
AES-CBC round-trip OK


## Demo — AES Cipher (CTR mode)
No padding required — CTR turns AES into a stream cipher. Output length == input length.


In [25]:
import os
from Crypto.Cipher import AES

plaintext = b"HELLO WORLD"
raw_key   = os.urandom(16)                 # AES-128
nonce     = os.urandom(8)                  # 8-byte nonce (CTR default)

cfg = AESConfig(key=raw_key, mode=AES.MODE_CTR, nonce=nonce)

ciphertext = cipher_encrypt(plaintext, cfg)
recovered  = cipher_decrypt(ciphertext, cfg)

print(f"Plaintext : {plaintext}")
print(f"Ciphertext: {ciphertext.hex()}  (len={len(ciphertext)})")
print(f"Recovered : {recovered}")
assert recovered == plaintext
print("AES-CTR round-trip OK")


Plaintext : b'HELLO WORLD'
Ciphertext: 82047636e6310825a9aba9  (len=11)
Recovered : b'HELLO WORLD'
AES-CTR round-trip OK


## Demo — AES Cipher (CFB mode)
No padding required — stream-like. Output length == input length.


In [26]:
import os
from Crypto.Cipher import AES

plaintext = b"HELLO WORLD"
raw_key   = os.urandom(16)                 # AES-128
iv        = os.urandom(16)                 # random IV

cfg = AESConfig(key=raw_key, mode=AES.MODE_CFB, iv=iv)

ciphertext = cipher_encrypt(plaintext, cfg)
recovered  = cipher_decrypt(ciphertext, cfg)

print(f"Plaintext : {plaintext}")
print(f"Ciphertext: {ciphertext.hex()}  (len={len(ciphertext)})")
print(f"Recovered : {recovered}")
assert recovered == plaintext
print("AES-CFB round-trip OK")


Plaintext : b'HELLO WORLD'
Ciphertext: d15afa6421c97bd380b82f  (len=11)
Recovered : b'HELLO WORLD'
AES-CFB round-trip OK
